# 🔧 GalFitM Inputs — z01_lsst en **DP2** (galaxias resueltas + parámetros Sérsic de LSST)

**Cambios respecto a v5 (DP1)**
- **DP2**: Butler `dp2`, skymap `lsst_cells_v2`. El `deep_coadd` ahora es un `CellCoadd` (paquete `lsst.images`): se lee sólo el recorte con `parameters={'bbox': ...}` y la PSF con el componente `deep_coadd.psf` (`compute_kernel_image`).
- **Selección por tamaño vs PSF**: se conserva una galaxia si `re_mayor > FACTOR_PSF × FWHM_PSF`. La FWHM se mide con momentos adaptativos HSM (GalSim) sobre la PSF del coadd en la posición de la galaxia, en cada banda (por defecto se usa la peor).
- **Valores iniciales de GalFitM desde el ajuste Sérsic de LSST**: centro, magnitud por banda, `re`, `n`, `q`, `PA`.
- **Solo se descargan galaxias con g, r e i** (`BANDAS_OBLIGATORIAS`): se piden primero sus PSF y, si falta alguna o su stamp está vacío (> 30 % de píxeles en 0), la galaxia se descarta sin bajar el resto.
- **Se ajustan TODAS las bandas con coadd en DP2** (no sólo las que tienen `sersicFlux`). Magnitud inicial: `sersicFlux` → `cModelMag` → banda vecina en λ.

**Flujo:** cambiar `LOTE` → correr todo → bajar el tar.gz → `CONFIRMAR_BORRADO = True` → siguiente lote.

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Circle
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import gaussian_filter
import os, shutil, warnings, subprocess
warnings.filterwarnings('ignore')
import sep
import galsim

from lsst.daf.butler import Butler
from lsst.images import Box
import lsst.geom as geom

# ── DP2 ───────────────────────────────────────────────────────────────
REPO         = 'dp2'
COLLECTION   = 'dp2'
DS_TYPE      = 'deep_coadd'
SKYMAP_NAME  = 'lsst_cells_v2'
TODAS_BANDAS = ['u', 'g', 'r', 'i', 'z', 'y']
SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))

butler = Butler(REPO, collections=COLLECTION)
skymap = butler.get('skyMap', skymap=SKYMAP_NAME)
print('Butler DP2 listo ✓')

Butler DP2 listo ✓


In [2]:
# ── Parámetros de selección ─────────────────────────────────────────
ARCHIVO_CAT  = 'z01_lsst_min.parquet'
PIXEL_SCALE  = 0.2      # arcsec/px
MAGZERO      = 31.4     # AB para imágenes en nJy

FACTOR_PSF   = 1.0      # resuelta si re_mayor > FACTOR_PSF × FWHM_PSF
CRITERIO_FWHM = 'max'   # 'max' (peor banda), 'median', o una banda: 'r', 'i'...
FWHM_NOMINAL = 0.7      # arcsec — sólo para el pre-filtro (valor optimista)
CLASES_OK    = None     # p.ej. ['GALAXY', 'GALAXY?']; None = todas las clases
MIN_BANDAS_LSST = 3    # mínimo de filtros con ajuste LSST (≈ filtros con coadd en DP2)
MAX_CHI2     = None     # p.ej. 10 para descartar ajustes Sérsic malos; None = sin corte

# ── Carga y limpieza ────────────────────────────────────────────────
raw = pd.read_parquet(ARCHIVO_CAT)
cat = raw[raw['objectId'].notna()].copy()
cat = (cat.sort_values('_dist_arcsec')
          .drop_duplicates('objectId', keep='first')
          .sort_index())
cat['objectId'] = cat['objectId'].astype('int64')
print(f'Filas en parquet           : {len(raw)}')
print(f'objectId únicos            : {len(cat)}')

ok = (~cat['sersic_unknown_flag'].astype(bool) &
      ~cat['sersic_no_data_flag'].astype(bool) &
      np.isfinite(cat['sersic_reff_x']) & np.isfinite(cat['sersic_reff_y']) &
      np.isfinite(cat['sersic_rho'])    & np.isfinite(cat['sersic_index']) &
      (cat['sersic_reff_x'] > 0) & (cat['sersic_reff_y'] > 0))
if CLASES_OK is not None:
    ok &= cat['class'].isin(CLASES_OK)
if MAX_CHI2 is not None:
    ok &= cat['sersic_chi2_reduced'] < MAX_CHI2
cat = cat[ok].copy()
print(f'Con ajuste Sérsic válido   : {len(cat)}')


def params_lsst_a_galfit(df):
    """Ajuste Sérsic LSST (multiprofit) → parámetros iniciales de GalFitM.

    sersic_reff_x/y [px] y sersic_rho definen la elipse del radio efectivo
    como una 'covarianza' en el sistema de píxeles del coadd:
        Cxx = rx², Cyy = ry², Cxy = rho·rx·ry
    re (semieje mayor) y q salen de sus autovalores; el ángulo del eje mayor
    (desde +x, antihorario) se pasa a la convención de GALFIT
    (PA = 0 hacia +y, positivo antihorario):  PA = phi - 90.
    Como el stamp conserva la grilla de píxeles del coadd, no hace falta
    ninguna rotación adicional.
    """
    rx  = df['sersic_reff_x'].astype(float).values
    ry  = df['sersic_reff_y'].astype(float).values
    rho = df['sersic_rho'].astype(float).values
    cxx, cyy, cxy = rx**2, ry**2, rho * rx * ry
    t = np.sqrt(((cxx - cyy) / 2)**2 + cxy**2)
    a = np.sqrt((cxx + cyy) / 2 + t)
    b = np.sqrt(np.clip((cxx + cyy) / 2 - t, 0, None))
    phi = 0.5 * np.degrees(np.arctan2(2 * cxy, cxx - cyy))
    pa  = ((phi - 90.0) + 90.0) % 180.0 - 90.0            # (-90, 90]

    out = pd.DataFrame(index=df.index)
    out['gf_re_px']     = np.clip(a, 0.5, None)
    out['gf_re_arcsec'] = a * PIXEL_SCALE
    out['gf_q']         = np.clip(b / np.where(a > 0, a, 1), 0.05, 1.0)
    out['gf_pa']        = pa
    out['gf_n']         = np.clip(df['sersic_index'].astype(float).values, 0.3, 8.0)
    for bnd in TODAS_BANDAS:
        f = df[f'{bnd}_sersicFlux'].astype(float)
        out[f'gf_mag_{bnd}'] = np.where(f > 0, -2.5 * np.log10(f.where(f > 0)) + MAGZERO, np.nan)
    out['bandas_lsst'] = [
        [bnd for bnd in TODAS_BANDAS if np.isfinite(r[f'gf_mag_{bnd}'])]
        for _, r in out.iterrows()
    ]
    return out


cat = cat.join(params_lsst_a_galfit(cat))
# Posición: centro del ajuste Sérsic (fallback al centroide del objeto)
cat['ra']  = cat['sersic_ra'].where(np.isfinite(cat['sersic_ra']), cat['coord_ra'])
cat['dec'] = cat['sersic_dec'].where(np.isfinite(cat['sersic_dec']), cat['coord_dec'])

# ── Pre-filtro barato (la selección definitiva usa la PSF real) ─────
pre = (cat['gf_re_arcsec'] > FACTOR_PSF * FWHM_NOMINAL)
pre &= cat['bandas_lsst'].str.len() >= MIN_BANDAS_LSST   # ≥ N filtros con datos
cat = cat[pre].reset_index(drop=True)
print(f'Pasan pre-filtro (re > {FACTOR_PSF*FWHM_NOMINAL:.2f}\" y ≥{MIN_BANDAS_LSST} bandas) : {len(cat)}')
print()
print(cat[['objectId', 'z', 'gf_re_arcsec', 'gf_n', 'gf_q', 'gf_pa']].describe().T)
print()
print('Bandas con sersicFlux de LSST (sólo para magnitud inicial;')
print('GalFitM usa TODAS las bandas con coadd):')
print(cat['bandas_lsst'].str.len().value_counts().sort_index().to_string())

Filas en parquet           : 158769
objectId únicos            : 158259
Con ajuste Sérsic válido   : 158258
Pasan pre-filtro (re > 0.70" y ≥3 bandas) : 85170

                count          mean           std           min           25%  \
objectId      85170.0  7.797907e+17  1.174587e+16  7.348561e+17  7.755706e+17   
z             85170.0  6.536293e-02  2.328559e-02  1.034000e-05  4.969425e-02   
gf_re_arcsec  85170.0  1.950515e+00  1.220754e+00  7.000147e-01  1.247037e+00   
gf_n          85170.0  1.946231e+00  1.384773e+00  5.000000e-01  9.185550e-01   
gf_q          85170.0  5.980104e-01  2.065023e-01  5.000000e-02  4.355071e-01   
gf_pa         85170.0  1.687061e-01  5.186693e+01 -8.999974e+01 -4.480917e+01   

                       50%           75%           max  
objectId      7.822522e+17  7.873201e+17  7.972578e+17  
z             6.919675e-02  8.440012e-02  1.000000e-01  
gf_re_arcsec  1.762306e+00  2.409938e+00  1.177424e+02  
gf_n          1.364440e+00  2.690513e+00  6.0

In [3]:
# ── Parámetros de selección ─────────────────────────────────────────
ARCHIVO_CAT  = 'z01_lsst_min.parquet'
PIXEL_SCALE  = 0.2      # arcsec/px
MAGZERO      = 31.4     # AB para imágenes en nJy

FACTOR_PSF   = 1.0      # resuelta si re_mayor > FACTOR_PSF × FWHM_PSF
CRITERIO_FWHM = 'max'   # 'max' (peor banda), 'median', o una banda: 'r', 'i'...
FWHM_NOMINAL = 0.7      # arcsec — sólo para el pre-filtro (valor optimista)
CLASES_OK    = None     # p.ej. ['GALAXY', 'GALAXY?']; None = todas las clases
MIN_BANDAS_LSST = 3    # mínimo de filtros con ajuste LSST (≈ filtros con coadd en DP2)
MAX_CHI2     = None     # p.ej. 10 para descartar ajustes Sérsic malos; None = sin corte

# ── Carga y limpieza ────────────────────────────────────────────────
raw = pd.read_parquet(ARCHIVO_CAT)
cat = raw[raw['objectId'].notna()].copy()
cat = (cat.sort_values('_dist_arcsec')
          .drop_duplicates('objectId', keep='first')
          .sort_index())
cat['objectId'] = cat['objectId'].astype('int64')
print(f'Filas en parquet           : {len(raw)}')
print(f'objectId únicos            : {len(cat)}')

ok = (~cat['sersic_unknown_flag'].astype(bool) &
      ~cat['sersic_no_data_flag'].astype(bool) &
      np.isfinite(cat['sersic_reff_x']) & np.isfinite(cat['sersic_reff_y']) &
      np.isfinite(cat['sersic_rho'])    & np.isfinite(cat['sersic_index']) &
      (cat['sersic_reff_x'] > 0) & (cat['sersic_reff_y'] > 0))
if CLASES_OK is not None:
    ok &= cat['class'].isin(CLASES_OK)
if MAX_CHI2 is not None:
    ok &= cat['sersic_chi2_reduced'] < MAX_CHI2
cat = cat[ok].copy()
print(f'Con ajuste Sérsic válido   : {len(cat)}')


def params_lsst_a_galfit(df):
    """Ajuste Sérsic LSST (multiprofit) → parámetros iniciales de GalFitM.

    sersic_reff_x/y [px] y sersic_rho definen la elipse del radio efectivo
    como una 'covarianza' en el sistema de píxeles del coadd:
        Cxx = rx², Cyy = ry², Cxy = rho·rx·ry
    re (semieje mayor) y q salen de sus autovalores; el ángulo del eje mayor
    (desde +x, antihorario) se pasa a la convención de GALFIT
    (PA = 0 hacia +y, positivo antihorario):  PA = phi - 90.
    Como el stamp conserva la grilla de píxeles del coadd, no hace falta
    ninguna rotación adicional.
    """
    rx  = df['sersic_reff_x'].astype(float).values
    ry  = df['sersic_reff_y'].astype(float).values
    rho = df['sersic_rho'].astype(float).values
    cxx, cyy, cxy = rx**2, ry**2, rho * rx * ry
    t = np.sqrt(((cxx - cyy) / 2)**2 + cxy**2)
    a = np.sqrt((cxx + cyy) / 2 + t)
    b = np.sqrt(np.clip((cxx + cyy) / 2 - t, 0, None))
    phi = 0.5 * np.degrees(np.arctan2(2 * cxy, cxx - cyy))
    pa  = ((phi - 90.0) + 90.0) % 180.0 - 90.0            # (-90, 90]

    out = pd.DataFrame(index=df.index)
    out['gf_re_px']     = np.clip(a, 0.5, None)
    out['gf_re_arcsec'] = a * PIXEL_SCALE
    out['gf_q']         = np.clip(b / np.where(a > 0, a, 1), 0.05, 1.0)
    out['gf_pa']        = pa
    out['gf_n']         = np.clip(df['sersic_index'].astype(float).values, 0.3, 8.0)
    for bnd in TODAS_BANDAS:
        f = df[f'{bnd}_sersicFlux'].astype(float)
        out[f'gf_mag_{bnd}'] = np.where(f > 0, -2.5 * np.log10(f.where(f > 0)) + MAGZERO, np.nan)
    out['bandas_lsst'] = [
        [bnd for bnd in TODAS_BANDAS if np.isfinite(r[f'gf_mag_{bnd}'])]
        for _, r in out.iterrows()
    ]
    return out


cat = cat.join(params_lsst_a_galfit(cat))
# Posición: centro del ajuste Sérsic (fallback al centroide del objeto)
cat['ra']  = cat['sersic_ra'].where(np.isfinite(cat['sersic_ra']), cat['coord_ra'])
cat['dec'] = cat['sersic_dec'].where(np.isfinite(cat['sersic_dec']), cat['coord_dec'])

# ── Pre-filtro barato (la selección definitiva usa la PSF real) ─────
pre = (cat['gf_re_arcsec'] > FACTOR_PSF * FWHM_NOMINAL)
pre &= cat['bandas_lsst'].str.len() >= MIN_BANDAS_LSST   # ≥ N filtros con datos
cat = cat[pre].reset_index(drop=True)
print(f'Pasan pre-filtro (re > {FACTOR_PSF*FWHM_NOMINAL:.2f}\" y ≥{MIN_BANDAS_LSST} bandas) : {len(cat)}')
print()
print(cat[['objectId', 'z', 'gf_re_arcsec', 'gf_n', 'gf_q', 'gf_pa']].describe().T)
print()
print('Bandas con sersicFlux de LSST (sólo para magnitud inicial;')
print('GalFitM usa TODAS las bandas con coadd):')
print(cat['bandas_lsst'].str.len().value_counts().sort_index().to_string())

Filas en parquet           : 158769
objectId únicos            : 158259
Con ajuste Sérsic válido   : 158258
Pasan pre-filtro (re > 0.70" y ≥3 bandas) : 85170

                count          mean           std           min           25%  \
objectId      85170.0  7.797907e+17  1.174587e+16  7.348561e+17  7.755706e+17   
z             85170.0  6.536293e-02  2.328559e-02  1.034000e-05  4.969425e-02   
gf_re_arcsec  85170.0  1.950515e+00  1.220754e+00  7.000147e-01  1.247037e+00   
gf_n          85170.0  1.946231e+00  1.384773e+00  5.000000e-01  9.185550e-01   
gf_q          85170.0  5.980104e-01  2.065023e-01  5.000000e-02  4.355071e-01   
gf_pa         85170.0  1.687061e-01  5.186693e+01 -8.999974e+01 -4.480917e+01   

                       50%           75%           max  
objectId      7.822522e+17  7.873201e+17  7.972578e+17  
z             6.919675e-02  8.440012e-02  1.000000e-01  
gf_re_arcsec  1.762306e+00  2.409938e+00  1.177424e+02  
gf_n          1.364440e+00  2.690513e+00  6.0

In [24]:
# ── CAMBIAR ESTO EN CADA CORRIDA ─────────────────────────────────────
LOTE       = 15   # ← 1, 2, 3, 4, 5, 6...
BATCH_SIZE = 800   # galaxias del catálogo pre-filtrado por lote
# ─────────────────────────────────────────────────────────────────────

SIZE_ASEC   = 120        # tamaño fijo del stamp (si STAMP_ADAPTATIVO = False)
STAMP_ADAPTATIVO = False # True → tamaño = STAMP_N_RE × re, acotado
STAMP_N_RE  = 20
SIZE_MIN, SIZE_MAX = 60, 240   # arcsec
OUTPUT_DIR  = 'galfitm_inputs'
CONV_BOX    = 200
FIJAR_ESTRUCTURA = False # True → re, n, q, PA fijos a los valores de LSST
BANDAS_AJUSTE = TODAS_BANDAS   # GalFitM usa todas estas que tengan coadd + PSF
MIN_BANDAS    = 3              # descartar galaxias con menos bandas que esto
BANDAS_OBLIGATORIAS = ['g', 'r', 'i']   # sin las tres, la galaxia NO se descarga
FRAC_VACIA_MAX = 0.30   # banda con > 30% de píxeles vacíos en el stamp = no cuenta


def npix_para(re_arcsec):
    size = SIZE_ASEC
    if STAMP_ADAPTATIVO:
        size = float(np.clip(STAMP_N_RE * re_arcsec, SIZE_MIN, SIZE_MAX))
    n = int(size / PIXEL_SCALE)
    return n + 1 if n % 2 == 0 else n


idx_inicio = (LOTE - 1) * BATCH_SIZE
idx_fin    = min(LOTE * BATCH_SIZE, len(cat))
cat_lote   = cat.iloc[idx_inicio:idx_fin].copy()

nombre_lote = f'galfitm_dp2_z01_lote{LOTE:02d}'
zip_path    = os.path.join(os.path.expanduser('~'), 'notebooks',
                           f'{nombre_lote}.tar.gz')
log_path    = os.path.join(OUTPUT_DIR, f'seleccion_psf_lote{LOTE:02d}.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

n_lotes_total = int(np.ceil(len(cat) / BATCH_SIZE))
print(f'=== LOTE {LOTE} de {n_lotes_total} ===')
print(f'Galaxias : {idx_inicio} → {idx_fin-1}  ({len(cat_lote)} candidatas)')
txt_stamp = 'adaptativo' if STAMP_ADAPTATIVO else f'{SIZE_ASEC} arcsec fijo'
print(f'Stamp    : {txt_stamp}')
print(f'Zip      : {zip_path}')

=== LOTE 15 de 107 ===
Galaxias : 11200 → 11999  (800 candidatas)
Stamp    : 120 arcsec fijo
Zip      : /home/gissel-montaguth/notebooks/galfitm_dp2_z01_lote15.tar.gz


In [25]:
LAMBDA_EFF = {'u': 3550, 'g': 4670, 'r': 6160,
              'i': 7470, 'z': 8920, 'y': 10200}


def ubicar(ra, dec):
    """Tract/patch 'propio' de la galaxia y su posición en píxeles del tract
    (el mismo sistema absoluto que usa el CellCoadd)."""
    sky = geom.SpherePoint(ra, dec, geom.degrees)
    ti  = skymap.findTract(sky)
    pi  = ti.findPatch(sky)
    pix = ti.getWcs().skyToPixel(sky)
    return sky, ti, pi, pix.getX(), pix.getY()


def data_id(ti, pi, band):
    return {'band': band, 'tract': ti.tract_id,
            'patch': pi.sequential_index, 'skymap': SKYMAP_NAME}


def fwhm_hsm(kernel):
    """FWHM (arcsec) de la PSF con momentos adaptativos HSM (como en el
    tutorial DP2 205.1). Fallback: momentos de segundo orden simples."""
    try:
        img = galsim.Image(np.ascontiguousarray(kernel, dtype=np.float64), scale=1.0)
        sigma = galsim.hsm.FindAdaptiveMom(img, guess_sig=3.0).moments_sigma
    except Exception:
        k = np.clip(kernel, 0, None); k = k / k.sum()
        yy, xx = np.indices(k.shape)
        xc, yc = (k * xx).sum(), (k * yy).sum()
        ixx = (k * (xx - xc)**2).sum(); iyy = (k * (yy - yc)**2).sum()
        ixy = (k * (xx - xc) * (yy - yc)).sum()
        sigma = (ixx * iyy - ixy**2) ** 0.25
    return float(sigma * SIGMA_TO_FWHM * PIXEL_SCALE)


def psf_banda(ti, pi, x, y, band):
    """Lee sólo el componente deep_coadd.psf (CellPointSpreadFunction) y
    devuelve (kernel normalizado, FWHM en arcsec). (None, NaN) si no hay coadd."""
    try:
        psf = butler.get(f'{DS_TYPE}.psf', dataId=data_id(ti, pi, band))
        k = np.asarray(psf.compute_kernel_image(x=x, y=y).array, dtype=np.float64)
        if not np.isfinite(k).all() or k.sum() <= 0:
            return None, np.nan
        k /= k.sum()
        return k, fwhm_hsm(k)
    except Exception:
        return None, np.nan


def fwhm_referencia(fwhms):
    vals = {b: v for b, v in fwhms.items() if np.isfinite(v)}
    if not vals:
        return np.nan
    if CRITERIO_FWHM == 'max':
        return max(vals.values())
    if CRITERIO_FWHM == 'median':
        return float(np.median(list(vals.values())))
    return vals.get(CRITERIO_FWHM, max(vals.values()))


def make_stamp(ti, pi, band, x, y, npix):
    """Lee del Butler SÓLO el recorte npix×npix alrededor de (x, y) (píxeles
    del tract). Lo que cae fuera del patch se rellena con NaN.
    Devuelve sci, var, header WCS y la posición exacta (1-indexada) de la galaxia."""
    half = npix // 2
    xc, yc = int(round(x)), int(round(y))
    x_start, y_start = xc - half, yc - half            # esquina del stamp pedido

    ob = pi.getOuterBBox()                             # bbox del patch (inclusive)
    x0c, x1c = max(x_start, ob.getMinX()), min(x_start + npix, ob.getMaxX() + 1)
    y0c, y1c = max(y_start, ob.getMinY()), min(y_start + npix, ob.getMaxY() + 1)
    if x1c <= x0c or y1c <= y0c:
        raise ValueError('stamp fuera del patch')

    sub = butler.get(DS_TYPE, dataId=data_id(ti, pi, band),
                     parameters={'bbox': Box.factory[y0c:y1c, x0c:x1c]})
    sci_c = np.asarray(sub.image.array, dtype=np.float32)
    var_c = np.asarray(sub.variance.array, dtype=np.float32)

    sci = np.full((npix, npix), np.nan, dtype=np.float32)
    var = np.full((npix, npix), np.nan, dtype=np.float32)
    ox, oy = x0c - x_start, y0c - y_start
    h, w = sci_c.shape
    sci[oy:oy+h, ox:ox+w] = sci_c
    var[oy:oy+h, ox:ox+w] = var_c

    x0 = x - x_start + 1.0                             # convención FITS/GALFIT
    y0 = y - y_start + 1.0

    wcs_t = ti.getWcs()
    pc    = geom.Point2D(xc, yc)
    crval = wcs_t.pixelToSky(pc)
    cd    = wcs_t.getCdMatrix(pc)
    hdr = {
        'NAXIS1': npix, 'NAXIS2': npix,
        'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
        'CRPIX1': half + 1,   'CRPIX2': half + 1,
        'CRVAL1': crval.getRa().asDegrees(),
        'CRVAL2': crval.getDec().asDegrees(),
        'CD1_1': cd[0,0], 'CD1_2': cd[0,1],
        'CD2_1': cd[1,0], 'CD2_2': cd[1,1],
    }
    return sci, var, hdr, x0, y0


def sigma_from_variance(var_arr):
    sigma = np.sqrt(np.abs(var_arr))
    med   = np.nanmedian(sigma[sigma > 0])
    return np.where(np.isnan(sigma) | (sigma <= 0), med, sigma).astype(np.float32)


def generar_mascara(sci_path, mask_path, x0, y0, radio_px=20, threshold=3.0):
    """Enmascara fuentes SEP excepto las que caen dentro de radio_px de la galaxia,
    y también los píxeles sin datos (NaN → 0 en el stamp)."""
    if not os.path.exists(sci_path):
        return None
    data = np.ascontiguousarray(fits.getdata(sci_path).astype(np.float64))
    sin_datos = (data == 0.0)
    bkg  = sep.Background(data, mask=sin_datos)
    data_sub = data - bkg.back()
    mask = sin_datos.astype(np.int16)
    try:
        objects = sep.extract(data_sub, threshold, err=bkg.rms(), mask=sin_datos)
    except Exception:
        fits.writeto(mask_path, mask, overwrite=True)
        return mask
    cx, cy = x0 - 1.0, y0 - 1.0
    yy, xx = np.ogrid[:data.shape[0], :data.shape[1]]
    for obj in objects:
        ox, oy = obj['x'], obj['y']
        if np.hypot(ox - cx, oy - cy) < radio_px:
            continue
        a, b = max(obj['a'], 1.0), max(obj['b'], 1.0)
        cos_t, sin_t = np.cos(obj['theta']), np.sin(obj['theta'])
        ellipse = (((xx-ox)*cos_t + (yy-oy)*sin_t)/a)**2 + \
                  (((xx-ox)*sin_t - (yy-oy)*cos_t)/b)**2
        mask[ellipse <= 9] = 1
    fits.writeto(mask_path, mask, overwrite=True)
    return mask


def estimar_cielo_3sigma(sci_path, borde_frac=0.15):
    if not os.path.exists(sci_path):
        return 0.0
    data = fits.getdata(sci_path).astype(np.float64)
    ny, nx = data.shape
    borde  = max(3, int(min(ny, nx) * borde_frac))
    m = np.zeros((ny, nx), dtype=bool)
    m[:borde,:] = True;  m[-borde:,:] = True
    m[:,:borde] = True;  m[:,-borde:] = True
    pixeles = data[m & np.isfinite(data) & (data != 0.0)]
    if len(pixeles) < 10:
        return 0.0
    _, sky_med, _ = sigma_clipped_stats(pixeles, sigma=3.0)
    return float(sky_med)


def write_feedme(row, bandas, npix, x0, y0, outdir, conv_box):
    """Feedme de GalFitM con valores iniciales del ajuste Sérsic de LSST."""
    oid  = row['objectId']
    gdir = os.path.join(outdir, str(oid))
    nb   = len(bandas)

    def csl(*vals): return ','.join(str(v) for v in vals)
    def rep(val, n): return csl(*[val]*n)

    sci_files  = csl(*[f'{gdir}/{oid}_sci_{b}.fits' for b in bandas])
    sig_files  = csl(*[f'{gdir}/{oid}_sig_{b}.fits' for b in bandas])
    psf_files  = csl(*[f'{gdir}/{oid}_psf_{b}.fits' for b in bandas])
    out_file   = f'{gdir}/{oid}_galfitm_out.fits'
    mask_path  = f'{gdir}/{oid}_mask.fits'
    mask_files = csl(*[mask_path]*nb) if os.path.exists(mask_path) \
                 else csl(*['none']*nb)

    # Magnitud inicial por banda:
    #   1) sersicFlux de LSST  2) cModelMag  3) la banda más cercana en λ que tenga valor
    def mag_valida(m):
        return m is not None and np.isfinite(m) and 5 < m < 35
    mags = {}
    for b in TODAS_BANDAS:
        m = row.get(f'gf_mag_{b}', np.nan)
        if not mag_valida(m):
            m = row.get(f'{b}_cModelMag', np.nan)
        mags[b] = m if mag_valida(m) else np.nan
    con_valor = [b for b in TODAS_BANDAS if np.isfinite(mags[b])]
    mag_init = []
    for b in bandas:
        m = mags[b]
        if not np.isfinite(m):
            m = mags[min(con_valor, key=lambda c: abs(LAMBDA_EFF[c] - LAMBDA_EFF[b]))] \
                if con_valor else 18.0
        mag_init.append(f'{m:.4f}')

    re_init = float(np.clip(row['gf_re_px'], 0.5, npix / 2))
    n_init, q_init, pa_init = row['gf_n'], row['gf_q'], row['gf_pa']

    dof_pos = 1                       # centro común a todas las bandas
    dof_mag = nb                      # magnitud libre en cada banda
    dof_str = 0 if FIJAR_ESTRUCTURA else min(2, nb)   # re, n, q, PA

    sky_vals = [
        f'{estimar_cielo_3sigma(os.path.join(gdir, f"{oid}_sci_{b}.fits")):.6f}'
        for b in bandas
    ]

    lines = [
        f'# GalFitM — objectId={oid}  z={row["z"]:.4f}  bandas={bandas}',
        f'# RA={row["ra"]:.6f}  Dec={row["dec"]:.6f}  (centro Sérsic LSST)',
        f'# LSST Sérsic: re={row["gf_re_arcsec"]:.3f}"  n={n_init:.3f}  '
        f'q={q_init:.3f}  PA={pa_init:.2f}  chi2r={row["sersic_chi2_reduced"]:.2f}',
        f'# FWHM_PSF_ref={row["fwhm_ref"]:.3f}"  re/FWHM={row["gf_re_arcsec"]/row["fwhm_ref"]:.2f}',
        '',
        f'A) {sci_files}',
        f'A1) {csl(*bandas)}',
        f'A2) {csl(*[LAMBDA_EFF[b] for b in bandas])}',
        f'B) {out_file}',
        f'C) {sig_files} 1.0',
        f'D) {psf_files}',
        f'E) 1', f'F) {mask_files}', f'G) none',
        f'H) 1  {npix}  1  {npix}',
        f'I) {min(conv_box, npix)}  {min(conv_box, npix)}',
        f'J) {rep(MAGZERO, nb)}',
        f'K) {PIXEL_SCALE} {PIXEL_SCALE}',
        f'O) regular', f'P) 0', f'U) 0',
        f'W) input,model,residual', '',
        '#----------Galaxy-----------',
        ' 0) sersic',
        f' 1) {rep(f"{x0:.3f}", nb)} {dof_pos}',
        f' 2) {rep(f"{y0:.3f}", nb)} {dof_pos}',
        f' 3) {csl(*mag_init)} {dof_mag}',
        f' 4) {rep(f"{re_init:.4f}", nb)} {dof_str}',
        f' 5) {rep(f"{n_init:.4f}", nb)} {dof_str}',
        f' 9) {rep(f"{q_init:.4f}", nb)} {dof_str}',
        f'10) {rep(f"{pa_init:.4f}", nb)} {dof_str}',
        ' Z) 0', '',
        '#----------Sky-----------',
        ' 0) sky',
        f' 1) {csl(*sky_vals)} 1',
        ' 2) 0.000 0', ' 3) 0.000 0', ' Z) 0', '',
    ]
    fpath = os.path.join(gdir, f'{oid}.feedme')
    with open(fpath, 'w') as f:
        f.write('\n'.join(lines))
    return fpath


print('Funciones definidas ✓')

Funciones definidas ✓


In [ ]:
print(f'=== PROCESANDO LOTE {LOTE} ({len(cat_lote)} candidatas) ===\n')

log = []
aceptadas = []

for pos, (idx, row) in enumerate(cat_lote.iterrows(), start=1):
    oid = row['objectId']
    sky, ti, pi, xg, yg = ubicar(row['ra'], row['dec'])
    entrada = {'objectId': oid, 'tract': ti.tract_id, 'patch': pi.sequential_index,
               're_arcsec': row['gf_re_arcsec'], 'n': row['gf_n'],
               'q': row['gf_q'], 'pa': row['gf_pa']}

    # 1) PSF de cada banda con ajuste LSST (sólo el componente .psf) → FWHM
    # Primero las bandas obligatorias: si falta una, no se piden las demás
    kernels, fwhms = {}, {}
    orden = BANDAS_OBLIGATORIAS + [b for b in BANDAS_AJUSTE if b not in BANDAS_OBLIGATORIAS]
    for b in orden:
        kernels[b], fwhms[b] = psf_banda(ti, pi, xg, yg, b)
        if b in BANDAS_OBLIGATORIAS and kernels[b] is None:
            break
    # (solo las consultadas: si falta g, r e i ni se piden)
    faltan = [b for b in BANDAS_OBLIGATORIAS if b in kernels and kernels[b] is None]
    n_cob = sum(np.isfinite(v) for v in fwhms.values())
    entrada['n_bandas_coadd'] = n_cob
    for b in TODAS_BANDAS:
        entrada[f'fwhm_{b}'] = fwhms.get(b, np.nan)
    fref = fwhm_referencia(fwhms)
    entrada['fwhm_ref'] = fref
    entrada['re_sobre_fwhm'] = row['gf_re_arcsec'] / fref if np.isfinite(fref) else np.nan

    if faltan:
        entrada['estado'] = 'sin_' + ''.join(faltan)        # p. ej. 'sin_g', 'sin_gr'
        log.append(entrada); continue
    if not np.isfinite(fref) or n_cob < MIN_BANDAS:
        entrada['estado'] = 'sin_cobertura'
        log.append(entrada); continue
    if row['gf_re_arcsec'] <= FACTOR_PSF * fref:
        entrada['estado'] = 'no_resuelta'
        log.append(entrada); continue

    # 2) Galaxia resuelta → recorte (sólo los píxeles necesarios), sigma y PSF
    row = row.copy(); row['fwhm_ref'] = fref
    gdir = os.path.join(OUTPUT_DIR, str(oid))
    os.makedirs(gdir, exist_ok=True)
    npix = npix_para(row['gf_re_arcsec'])
    x0 = y0 = None

    bandas_gal = [b for b in BANDAS_AJUSTE if kernels.get(b) is not None]
    for band in bandas_gal:
        try:
            sci, var, hdr, xb, yb = make_stamp(ti, pi, band, xg, yg, npix)
            if x0 is None:
                x0, y0 = xb, yb
            sci_clean = np.where(np.isfinite(sci), sci, 0.0).astype(np.float32)
            frac_vacia = float(np.mean(sci_clean == 0))
            entrada[f'vacia_{band}'] = round(frac_vacia, 3)
            if frac_vacia > FRAC_VACIA_MAX:              # banda (casi) sin datos: no se guarda
                print(f'  ⚠️  oid={oid} banda={band}: {100*frac_vacia:.0f}% vacía, se omite')
                continue
            sigma     = sigma_from_variance(var)

            h = fits.Header()
            for k, v in hdr.items(): h[k] = v
            h['BAND'] = band;  h['MAGZERO'] = MAGZERO
            h['OBJECT'] = str(oid);  h['PIXSCALE'] = PIXEL_SCALE
            h['TRACT'] = ti.tract_id;  h['PATCH'] = pi.sequential_index
            h['FWHMPSF'] = (fwhms[band], 'arcsec, PSF coadd (HSM)')
            fits.writeto(os.path.join(gdir, f'{oid}_sci_{band}.fits'), sci_clean, header=h, overwrite=True)
            fits.writeto(os.path.join(gdir, f'{oid}_sig_{band}.fits'), sigma,     header=h, overwrite=True)

            h_psf = fits.Header()
            h_psf['BAND'] = band;  h_psf['OBJECT'] = str(oid)
            h_psf['PIXSCALE'] = PIXEL_SCALE
            h_psf['PSF_RA'] = row['ra'];  h_psf['PSF_DEC'] = row['dec']
            h_psf['FWHMPSF'] = fwhms[band]
            fits.writeto(os.path.join(gdir, f'{oid}_psf_{band}.fits'),
                         kernels[band].astype(np.float32), header=h_psf, overwrite=True)
        except Exception as e:
            print(f'  ❌ oid={oid} banda={band}: {str(e)[:80]}')

    # 3) Máscara SEP (en r si existe; si no, en la primera banda disponible) + feedme
    bandas_ok = [b for b in bandas_gal if all(
        os.path.exists(os.path.join(gdir, f'{oid}_{t}_{b}.fits')) for t in ('sci', 'sig', 'psf'))]
    faltan_datos = [b for b in BANDAS_OBLIGATORIAS if b not in bandas_ok]
    if faltan_datos:                                     # g, r o i vacía o fallida
        entrada['estado'] = 'sin_datos_' + ''.join(faltan_datos)
        shutil.rmtree(gdir, ignore_errors=True)
        log.append(entrada); continue
    if bandas_ok and x0 is not None:
        b_mask = 'r' if 'r' in bandas_ok else bandas_ok[0]
        generar_mascara(os.path.join(gdir, f'{oid}_sci_{b_mask}.fits'),
                        os.path.join(gdir, f'{oid}_mask.fits'), x0, y0,
                        radio_px=max(20, 2 * row['gf_re_px']))
        write_feedme(row, bandas_ok, npix, x0, y0, OUTPUT_DIR, CONV_BOX)
        entrada.update(estado='ok', bandas=','.join(bandas_ok), npix=npix, x0=x0, y0=y0)
        aceptadas.append(oid)
    else:
        entrada['estado'] = 'error_fits'
        shutil.rmtree(gdir, ignore_errors=True)
    log.append(entrada)

    if pos % 10 == 0 or pos == len(cat_lote):
        print(f'  {pos}/{len(cat_lote)} revisadas — {len(aceptadas)} aceptadas')

df_log = pd.DataFrame(log)
df_log.to_csv(log_path, index=False)
print(f'\n✅ Lote {LOTE} completado')
print(df_log['estado'].value_counts().to_string())
print(f'Log: {log_path}')

=== PROCESANDO LOTE 15 (800 candidatas) ===

  10/800 revisadas — 8 aceptadas
  40/800 revisadas — 27 aceptadas
  80/800 revisadas — 51 aceptadas
  90/800 revisadas — 59 aceptadas
  100/800 revisadas — 66 aceptadas
  110/800 revisadas — 73 aceptadas
  120/800 revisadas — 80 aceptadas
  130/800 revisadas — 87 aceptadas
  140/800 revisadas — 94 aceptadas
  150/800 revisadas — 98 aceptadas
  160/800 revisadas — 107 aceptadas
  180/800 revisadas — 117 aceptadas
  200/800 revisadas — 129 aceptadas
  210/800 revisadas — 135 aceptadas
  220/800 revisadas — 142 aceptadas
  230/800 revisadas — 148 aceptadas
  240/800 revisadas — 153 aceptadas
  ❌ oid=793991981033341806 banda=g: Failure from formatter 'lsst.images.formatters.GenericFormatter' for dataset 019
  250/800 revisadas — 161 aceptadas
  280/800 revisadas — 176 aceptadas
  290/800 revisadas — 180 aceptadas
  330/800 revisadas — 194 aceptadas
  340/800 revisadas — 198 aceptadas
  350/800 revisadas — 205 aceptadas
  360/800 revisadas — 213

In [ ]:
# Sanity check: LSST Sérsic fit ellipse (re, q, PA) over the image
IDX_ACCEPTED = 0   # ← index among the accepted galaxies of this batch

oid  = aceptadas[IDX_ACCEPTED]
row  = cat_lote[cat_lote['objectId'] == oid].iloc[0]
lg   = df_log[df_log['objectId'] == oid].iloc[0]
gdir = os.path.join(OUTPUT_DIR, str(oid))
bnd  = 'r' if os.path.exists(os.path.join(gdir, f'{oid}_sci_r.fits')) else lg['bandas'].split(',')[0]

sci  = fits.getdata(os.path.join(gdir, f'{oid}_sci_{bnd}.fits')).astype(np.float32)
sig  = fits.getdata(os.path.join(gdir, f'{oid}_sig_{bnd}.fits')).astype(np.float32)
psf  = fits.getdata(os.path.join(gdir, f'{oid}_psf_{bnd}.fits')).astype(np.float32)
mf   = os.path.join(gdir, f'{oid}_mask.fits')
mask = fits.getdata(mf).astype(np.float32) if os.path.exists(mf) else np.zeros_like(sci)

def asinh_norm(img, lo=0.5, hi=99.5, stretch=0.5):
    v1, v2 = np.nanpercentile(img, [lo, hi])
    x = np.clip((img-v1)/(v2-v1+1e-12), 0, 1)
    return np.clip(np.arcsinh(x/stretch)/np.arcsinh(1.0/stretch), 0, 1)

cx, cy = lg['x0'] - 1, lg['y0'] - 1      # 0-based for matplotlib
re_px  = row['gf_re_px']
zoom   = max(25, 6 * re_px)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))

axes[0].imshow(asinh_norm(sci), cmap='gray', origin='lower')
axes[0].set_title(f'Science image, {bnd} band (full stamp)')

axes[1].imshow(asinh_norm(sci), cmap='gray', origin='lower')
# GALFIT PA (from +y, counter-clockwise) → matplotlib angle (from +x) = PA + 90
for k, col in [(1, 'cyan'), (3, 'yellow')]:
    axes[1].add_patch(Ellipse((cx, cy), 2*k*re_px, 2*k*re_px*row['gf_q'],
                              angle=row['gf_pa'] + 90, fill=False, color=col, lw=1.5))
axes[1].add_patch(Circle((cx, cy), lg['fwhm_ref'] / PIXEL_SCALE / 2,
                         fill=False, color='red', lw=1.5, ls='--'))
axes[1].set_xlim(cx - zoom, cx + zoom); axes[1].set_ylim(cy - zoom, cy + zoom)
axes[1].set_title('LSST Sérsic: 1 $R_e$ (cyan), 3 $R_e$ (yellow)\nPSF FWHM (red)')

axes[2].imshow(asinh_norm(sci), cmap='gray', origin='lower')
axes[2].imshow(mask, cmap='Reds', origin='lower', alpha=0.5, vmin=0, vmax=1)
axes[2].set_title(f'SEP mask\n{int(mask.sum()):,} px ({100*mask.sum()/mask.size:.1f}%)')

im3 = axes[3].imshow(np.log10(psf/psf.max()+1e-6), cmap='gray', origin='lower')
plt.colorbar(im3, ax=axes[3], fraction=0.046, label='log$_{10}$(PSF / max)')
axes[3].set_title(f'PSF, {bnd} band  ({psf.shape[0]}×{psf.shape[1]} px)')

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'sanity_lote{LOTE:02d}_{oid}.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
oids_lote = [str(o) for o in aceptadas if os.path.exists(os.path.join(OUTPUT_DIR, str(o)))]
extras = [os.path.basename(log_path)] + [f for f in os.listdir(OUTPUT_DIR)
                                          if f.startswith(f'sanity_lote{LOTE:02d}_')]

print(f'Comprimiendo {len(oids_lote)} carpetas + {len(extras)} archivos → {zip_path}')
cmd    = ['tar', '-czf', zip_path, '-C', OUTPUT_DIR] + oids_lote + extras
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f'✅ Comprimido: {os.path.getsize(zip_path) / 1e6:.0f} MB\n')
    print('=== COPIAR ESTE URL EN EL NAVEGADOR PARA DESCARGAR ===')
    print(f'https://gissel-montaguth.nb.data.lsst.cloud/nb/user/gissel-montaguth/files/notebooks/{nombre_lote}.tar.gz')
else:
    print(f'❌ Error al comprimir: {result.stderr}')

In [ ]:
# ⚠️ SOLO CAMBIAR A True DESPUÉS DE CONFIRMAR QUE BAJÓ EL TAR.GZ
CONFIRMAR_BORRADO = True

if CONFIRMAR_BORRADO:
    n = 0
    for oid in aceptadas:
        gdir = os.path.join(OUTPUT_DIR, str(oid))
        if os.path.exists(gdir):
            shutil.rmtree(gdir); n += 1
    for f in os.listdir(OUTPUT_DIR):
        if f.startswith(f'sanity_lote{LOTE:02d}_'):
            os.remove(os.path.join(OUTPUT_DIR, f))
    if os.path.exists(zip_path):
        os.remove(zip_path)
        print(f'✅ Borrado: {zip_path}')
    print(f'✅ {n} carpetas borradas de {OUTPUT_DIR}/  (el log CSV se conserva)')
    print(subprocess.run(['df', '-h', os.path.expanduser('~')], capture_output=True, text=True).stdout)
    print(f'→ Cambiá LOTE a {LOTE+1} y corré de nuevo desde la celda 3')
else:
    print('⚠️  Cambiá CONFIRMAR_BORRADO = True una vez que hayas bajado el tar')